In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from meta import *

df = pd.read_csv(savepath + r"/student_eda.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6378 entries, 0 to 6377
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6378 non-null   int64
 1   Attendance                  6378 non-null   int64
 2   Parental_Involvement        6378 non-null   int64
 3   Access_to_Resources         6378 non-null   int64
 4   Extracurricular_Activities  6378 non-null   int64
 5   Sleep_Hours                 6378 non-null   int64
 6   Previous_Scores             6378 non-null   int64
 7   Motivation_Level            6378 non-null   int64
 8   Internet_Access             6378 non-null   int64
 9   Tutoring_Sessions           6378 non-null   int64
 10  Family_Income               6378 non-null   int64
 11  Teacher_Quality             6378 non-null   int64
 12  School_Type                 6378 non-null   int64
 13  Peer_Influence              6378 non-null   int64
 14  Physical

In [2]:
df['Pass_Fail'] = (df['Exam_Score'] >= 70).astype(int)

print("Pass/Fail Dağılımı:")
print(df['Pass_Fail'].value_counts())
print("\nOranlar:")
print(df['Pass_Fail'].value_counts(normalize=True) * 100)

X = df.drop(['Exam_Score', 'Pass_Fail'], axis=1)
y = df['Pass_Fail']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nÖzelliklerin listesi:\n{X.columns.tolist()}")

Pass/Fail Dağılımı:
Pass_Fail
0    4797
1    1581
Name: count, dtype: int64

Oranlar:
Pass_Fail
0    75.211665
1    24.788335
Name: proportion, dtype: float64
X shape: (6378, 19)
y shape: (6378,)

Özelliklerin listesi:
['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")
print(f"\nEğitim setinde Pass/Fail oranı:\n{y_train.value_counts()}")
print(f"\nTest setinde Pass/Fail oranı:\n{y_test.value_counts()}")

Eğitim seti boyutu: (5102, 19)
Test seti boyutu: (1276, 19)

Eğitim setinde Pass/Fail oranı:
Pass_Fail
0    3837
1    1265
Name: count, dtype: int64

Test setinde Pass/Fail oranı:
Pass_Fail
0    960
1    316
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled ortalaması: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled standart sapması: {X_train_scaled.std():.6f}")

X_train_scaled ortalaması: 0.000000
X_train_scaled standart sapması: 1.000000


In [5]:
from sklearn.svm import SVC

model = SVC(kernel='rbf', probability=True, random_state=42)
model.fit(X_train_scaled, y_train)

print("\nSVM modeli başarıyla eğitildi!")

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

print("İlk 10 tahmin:")
print(f"Tahmin edilen sınıflar: {y_pred[:10]}")
print(f"Geçme olasılıkları: {y_pred_proba[:10, 1]}")

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("MODEL PERFORMANS METRİKLERİ")
print("=" * 50)
print(f"Doğruluk (Accuracy):    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Kesinlik (Precision):   {precision:.4f} ({precision*100:.2f}%)")
print(f"Duyarlılık (Recall):    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Skoru:               {f1:.4f}")
print("\n" + "=" * 50)
print("KARIŞIKLIK MATRİSİ (Confusion Matrix)")
print("=" * 50)
cm = confusion_matrix(y_test, y_pred)
print(f"Doğru Negatif (TN):  {cm[0, 0]}  | Yanlış Pozitif (FP): {cm[0, 1]}")
print(f"Yanlış Negatif (FN): {cm[1, 0]}  | Doğru Pozitif (TP):  {cm[1, 1]}")
print("\n" + "=" * 50)
print("DETAYLI SINIFLANDIRMA RAPORU")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Kaldı (0)', 'Geçti (1)']))

X_all_scaled = scaler.transform(X)
df['Predicted_Pass_Fail'] = model.predict(X_all_scaled)
df['Pass_Probability'] = model.predict_proba(X_all_scaled)[:, 1]

print("=" * 60)
print("TÜMEL SONUÇLAR - KAÇI GEÇTİ, KAÇI KALDI?")
print("=" * 60)

pass_count = (df['Predicted_Pass_Fail'] == 1).sum()
fail_count = (df['Predicted_Pass_Fail'] == 0).sum()
total_count = len(df)

pass_percent = (pass_count / total_count) * 100
fail_percent = (fail_count / total_count) * 100

print(f"Toplam Öğrenci: {total_count}")
print(f"Geçen:      {pass_count:3d} ({pass_percent:6.2f}%)")
print(f"Kalan:      {fail_count:3d} ({fail_percent:6.2f}%)")


SVM modeli başarıyla eğitildi!
İlk 10 tahmin:
Tahmin edilen sınıflar: [0 1 1 0 0 0 0 0 0 1]
Geçme olasılıkları: [1.64680521e-02 9.99999559e-01 9.95684820e-01 6.24907284e-07
 2.61581856e-03 5.68417270e-05 1.31913387e-05 2.25246863e-04
 2.64886428e-04 8.66713923e-01]
MODEL PERFORMANS METRİKLERİ
Doğruluk (Accuracy):    0.9718 (97.18%)
Kesinlik (Precision):   0.9575 (95.75%)
Duyarlılık (Recall):    0.9272 (92.72%)
F1 Skoru:               0.9421

KARIŞIKLIK MATRİSİ (Confusion Matrix)
Doğru Negatif (TN):  947  | Yanlış Pozitif (FP): 13
Yanlış Negatif (FN): 23  | Doğru Pozitif (TP):  293

DETAYLI SINIFLANDIRMA RAPORU
              precision    recall  f1-score   support

   Kaldı (0)       0.98      0.99      0.98       960
   Geçti (1)       0.96      0.93      0.94       316

    accuracy                           0.97      1276
   macro avg       0.97      0.96      0.96      1276
weighted avg       0.97      0.97      0.97      1276

TÜMEL SONUÇLAR - KAÇI GEÇTİ, KAÇI KALDI?
Toplam Öğrenc